In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("/kaggle/input/crop-yield-csv/crop_yield.csv")

target_col = "Yield_tons_per_hectare"

categorical_cols = [
    "Region",
    "Soil_Type",
    "Crop",
    "Fertilizer_Used",
    "Irrigation_Used",
    "Weather_Condition"
]

# =========================
# 2. TRAIN MODEL
# =========================
X = df.drop(columns=[target_col])
y = df[target_col]

X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
feature_columns = X.columns

X_train, _, y_train, _ = train_test_split(
    X, y, test_size=0.4, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

model = LinearRegression()
model.fit(X_train_scaled, y_train)

# =========================
# 3. YIELD → PRODUCTIVITY
# =========================
def yield_to_label(y):
    if y < 0.44:
        return "Low"
    elif y <= 0.78:
        return "Medium"
    else:
        return "High"

df["Yield_Label"] = df[target_col].apply(yield_to_label)
le = LabelEncoder()
le.fit(df["Yield_Label"])

# =========================
# 4. USER INPUT (ACTUAL INPUT)
# =========================
print("\nEnter crop details:\n")

user_data = {
    "Region": input("Region: "),
    "Soil_Type": input("Soil Type: "),
    "Crop": input("Crop: "),
    "Fertilizer_Used": input("Fertilizer Used (Yes/No): "),
    "Irrigation_Used": input("Irrigation Used (Yes/No): "),
    "Weather_Condition": input("Weather Condition: "),
    "Rainfall_mm": float(input("Rainfall (mm): ")),
    "Temperature_Celsius": float(input("Temperature (C): ")),
}

# =========================
# 5. PREDICTION
# =========================
user_df = pd.DataFrame([user_data])
user_df = pd.get_dummies(user_df)
user_df = user_df.reindex(columns=feature_columns, fill_value=0)

user_scaled = scaler.transform(user_df)

predicted_yield = model.predict(user_scaled)[0]
productivity = yield_to_label(predicted_yield)

encoded = le.transform([productivity])[0]
decoded = le.inverse_transform([encoded])[0]

# =========================
# 6. OUTPUT
# =========================
print("\n--- Prediction Result ---")
print(f"Productivity Category : {decoded}")
print("R2 Score:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))


Enter crop details:



Region:  south
Soil Type:  chalky
Crop:  rice
Fertilizer Used (Yes/No):  yes
Irrigation Used (Yes/No):  yes
Weather Condition:  rainy
Rainfall (mm):  980.5379541
Temperature (C):  37.26346848



--- Prediction Result ---
Productivity Category : High
R2 Score: 0.9130250552995817
MAE: 0.3992490532418838
